# 04 — Render

**Purpose:** Build Plotly charts from model outputs and export a static HTML dashboard to `docs/` for GitHub Pages.

## Charts
| Chart | Data |
|-------|------|
| Yield curve spread (10y-2y) time series | `indicators.parquet` |
| Recession probability gauge + history | `indicators.parquet` |
| Inflation regime heatmap | `indicators.parquet` |
| Global growth pulse bar chart | `indicators.parquet` |
| Risk-on / risk-off dial | `latest_snapshot.json` |
| Key metrics header | `latest_snapshot.json` |

## Outputs
- `docs/index.html` — self-contained HTML dashboard (Plotly CDN)

## Papermill Parameters
- `run_date` — ISO date string injected by the GitHub Actions workflow

In [ ]:
run_date = None

In [ ]:
# Mount Google Drive for persistent storage (Colab only)
try:
    from google.colab import drive
    from pathlib import Path
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/macro-dashboard/data")
    DRIVE_DOCS = Path("/content/drive/MyDrive/macro-dashboard/docs")
    DRIVE_DOCS.mkdir(parents=True, exist_ok=True)
    _IN_COLAB = True
    print("Drive mounted.")
except Exception:
    _IN_COLAB = False
    print("Not in Colab — using local directories.")

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "plotly", "pyarrow"])
print("Packages ready.")

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUTPUTS_DIR = DRIVE_DATA / "outputs" if _IN_COLAB else Path("data/outputs")
DOCS_DIR    = DRIVE_DOCS             if _IN_COLAB else Path("docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUTPUTS_DIR : {OUTPUTS_DIR}")
print(f"DOCS_DIR    : {DOCS_DIR}")

In [ ]:
ind  = pd.read_parquet(OUTPUTS_DIR / "indicators.parquet")
snap = json.loads((OUTPUTS_DIR / "latest_snapshot.json").read_text())

ind.index = pd.to_datetime(ind.index)
# Trim to post-1990 for cleaner charts
ind = ind[ind.index >= "1990-01-01"]

print("Indicators loaded:", ind.shape, "| as_of:", snap["as_of"])
print(json.dumps(snap, indent=2))

In [ ]:
# -- Load US time-series for additional charts -------------------------
import math as _math

_us_path = OUTPUTS_DIR.parent / 'processed' / 'us_series.parquet'
try:
    _us_all = pd.read_parquet(_us_path)
    _us_all.index = pd.to_datetime(_us_all.index)
    us = _us_all[_us_all.index >= '1990-01-01'].copy()

    def _us_latest(col):
        try:
            s = _us_all[col].dropna()
            v = float(s.iloc[-1])
            return None if _math.isnan(v) else round(v, 4)
        except Exception:
            return None

    credit_spread_val = _us_latest('credit_spread')
    real_rate_val     = _us_latest('real_rate_10y')
    fed_funds_val     = _us_latest('fed_funds')
    unrate_val        = _us_latest('unrate')
    m2_yoy_val        = _us_latest('m2_yoy_pct')
    cpi_yoy_val       = _us_latest('cpi_yoy_pct')
    _us_ok = True
    print(f'US series: {_us_all.shape}')
    print(f'  credit={credit_spread_val} bps | real={real_rate_val}%'
          f' | fed={fed_funds_val}% | unrate={unrate_val}%')
except FileNotFoundError:
    print('WARNING: us_series.parquet not found')
    us = pd.DataFrame()
    credit_spread_val = real_rate_val = fed_funds_val = None
    unrate_val = m2_yoy_val = cpi_yoy_val = None
    _us_ok = False


In [ ]:
DARK = "plotly_dark"
CYAN, ORANGE, RED, GREEN, GREY = "#00bcd4", "#ff9800", "#ef5350", "#66bb6a", "#78909c"

# ── 1. Yield Curve Spread ──────────────────────────────────────────────────
fig_curve = go.Figure()
spread = ind["yield_spread_10y2y"].dropna()
fig_curve.add_trace(go.Scatter(
    x=spread.index, y=spread.values,
    mode="lines", name="10y–2y Spread",
    line=dict(color=CYAN, width=1.5),
    fill="tozeroy",
    fillcolor="rgba(0,188,212,0.08)"
))
fig_curve.add_hline(y=0, line=dict(color=RED, dash="dash", width=1))
fig_curve.update_layout(
    template=DARK, title="US Yield Curve Spread (10y – 2y)",
    yaxis_title="% points", height=350,
    margin=dict(l=50, r=20, t=50, b=40)
)

# ── 2. Recession Probability ───────────────────────────────────────────────
fig_rec = go.Figure()
rec = ind["recession_prob"].dropna()
fig_rec.add_trace(go.Scatter(
    x=rec.index, y=rec.values * 100,
    mode="lines", name="Recession Prob",
    line=dict(color=ORANGE, width=1.5),
    fill="tozeroy", fillcolor="rgba(255,152,0,0.1)"
))
fig_rec.add_hline(y=30, line=dict(color=RED, dash="dot", width=1),
                  annotation_text="30% threshold", annotation_position="bottom right")
fig_rec.update_layout(
    template=DARK, title="US Recession Probability — 12m Ahead (Estrella-Mishkin)",
    yaxis_title="%", yaxis_range=[0, 100], height=350,
    margin=dict(l=50, r=20, t=50, b=40)
)

# ── 3. Inflation Regime ────────────────────────────────────────────────────
fig_inf = go.Figure()
zscore = ind["inflation_zscore"].dropna()
colors = [RED if v > 1.5 else ORANGE if v > 0.5 else GREEN if v < -0.5 else GREY
          for v in zscore.values]
fig_inf.add_trace(go.Bar(
    x=zscore.index, y=zscore.values,
    marker_color=colors, name="Inflation Z-Score"
))
fig_inf.add_hline(y=1.5,  line=dict(color=RED,   dash="dot", width=1))
fig_inf.add_hline(y=-0.5, line=dict(color=GREEN, dash="dot", width=1))
fig_inf.update_layout(
    template=DARK, title="US Inflation Regime — CPI Z-Score vs 20yr Rolling Avg",
    yaxis_title="Z-Score", height=350,
    margin=dict(l=50, r=20, t=50, b=40)
)

# ── 4. Risk Score Gauge ────────────────────────────────────────────────────
risk_val = snap["risk_score"] or 0
fig_gauge = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=risk_val,
    delta={"reference": 0},
    title={"text": "US Risk-On / Risk-Off Score<br><span style='font-size:11px;color:#78909c'>Credit spread · Real rate · Yield curve</span>", "font": {"size": 14}},
    gauge={
        "axis": {"range": [-1, 1], "tickwidth": 1},
        "bar": {"color": GREEN if risk_val > 0 else RED},
        "steps": [
            {"range": [-1, -0.3], "color": "rgba(239,83,80,0.3)"},
            {"range": [-0.3, 0.3], "color": "rgba(120,144,156,0.2)"},
            {"range": [0.3, 1],   "color": "rgba(102,187,106,0.3)"},
        ],
        "threshold": {"line": {"color": "white", "width": 2}, "value": risk_val}
    }
))
fig_gauge.update_layout(template=DARK, height=350,
                        margin=dict(l=30, r=30, t=60, b=30))

# ── 5. Key Metrics Card ────────────────────────────────────────────────────
regime_labels = {-1: "Low", 0: "Normal", 1: "Elevated", 2: "Very High"}
metrics_html = f"""
<div style="font-family:monospace;background:#1e1e2e;color:#cdd6f4;
            padding:24px;border-radius:8px;line-height:2">
  <h2 style="color:#89b4fa;margin-top:0">Global Macro Snapshot — {snap['as_of']}</h2>
  <table style="width:100%;border-collapse:collapse">
    <tr><td>🇺🇸 Yield Spread 10y–2y</td><td style="color:{GREEN if (snap['yield_spread_10y2y'] or 0)>0 else RED}">
        {snap['yield_spread_10y2y']:+.2f}%</td></tr>
    <tr><td>🇺🇸 Yield Spread 10y–3m</td><td style="color:{GREEN if (snap['yield_spread_10y3m'] or 0)>0 else RED}">
        {snap['yield_spread_10y3m']:+.2f}%</td></tr>
    <tr><td>🇺🇸 Inversion Signal</td><td>{'🔴 YES' if snap['inversion_signal'] else '🟢 NO'}
        ({snap['months_inverted']} months)</td></tr>
    <tr><td>🇺🇸 Recession Probability</td><td style="color:{RED if (snap['recession_prob'] or 0)>0.3 else ORANGE}">
        {(snap['recession_prob'] or 0)*100:.1f}%</td></tr>
    <tr><td>🇺🇸 Inflation Regime</td><td>{regime_labels.get(snap['inflation_regime'], 'Normal')}
        (z={snap['inflation_zscore']})</td></tr>
    <tr><td>🌍 Global Growth Pulse</td><td>{f"{snap['global_growth_pulse']:.1f}%" if snap['global_growth_pulse'] else 'N/A'}</td></tr>
    <tr><td>🇺🇸 Risk Score</td><td style="color:{GREEN if (snap['risk_score'] or 0)>0 else RED}">
        {snap['risk_score']:+.2f} ({'Risk-On' if (snap['risk_score'] or 0)>0 else 'Risk-Off'})</td></tr>
  </table>
</div>"""

print("All figures built.")
fig_curve.show()
fig_rec.show()
fig_inf.show()
fig_gauge.show()

In [ ]:
# -- Additional charts: Credit Spread / Real Rate / Fed Funds / M2 ----
if _us_ok and not us.empty:
    fig_credit = go.Figure()
    if 'credit_spread' in us.columns:
        _cs = us['credit_spread'].dropna()
        fig_credit.add_trace(go.Scatter(
            x=_cs.index, y=_cs.values, mode='lines', name='HY Spread',
            line=dict(color='#f38ba8', width=1.5),
            fill='tozeroy', fillcolor='rgba(243,139,168,0.08)'
        ))
    fig_credit.update_layout(
        template=DARK, title='US High-Yield Credit Spread (ICE BofA OAS)',
        yaxis_title='bps', height=350, margin=dict(l=50, r=20, t=50, b=40)
    )

    fig_real = go.Figure()
    if 'real_rate_10y' in us.columns:
        _rr = us['real_rate_10y'].dropna()
        fig_real.add_trace(go.Bar(
            x=_rr.index, y=_rr.values,
            marker_color=[GREEN if v >= 0 else RED for v in _rr.values],
            name='Real 10y Rate'
        ))
        fig_real.add_hline(y=0, line=dict(color=GREY, dash='dash', width=1))
    fig_real.update_layout(
        template=DARK, title='US Real 10-Year Rate (Nominal − CPI YoY)',
        yaxis_title='% pts', height=350, margin=dict(l=50, r=20, t=50, b=40)
    )

    fig_fed = go.Figure()
    if 'fed_funds' in us.columns:
        _ff = us['fed_funds'].dropna()
        fig_fed.add_trace(go.Scatter(
            x=_ff.index, y=_ff.values, mode='lines', name='Fed Funds Rate',
            line=dict(color='#a6e3a1', width=2)
        ))
    if 't2y' in us.columns:
        _t2 = us['t2y'].dropna()
        fig_fed.add_trace(go.Scatter(
            x=_t2.index, y=_t2.values, mode='lines', name='2-Year Yield',
            line=dict(color=CYAN, width=1.5, dash='dot')
        ))
    fig_fed.update_layout(
        template=DARK, title='Fed Funds Rate vs 2-Year Treasury Yield',
        yaxis_title='%', height=350, margin=dict(l=50, r=20, t=50, b=40),
        legend=dict(orientation='h', x=0, y=1.12, font=dict(size=11))
    )

    fig_m2 = go.Figure()
    if 'm2_yoy_pct' in us.columns:
        _m2 = us['m2_yoy_pct'].dropna()
        fig_m2.add_trace(go.Bar(
            x=_m2.index, y=_m2.values,
            marker_color=[GREEN if v >= 0 else RED for v in _m2.values],
            name='M2 YoY %'
        ))
    fig_m2.update_layout(
        template=DARK, title='US M2 Money Supply — YoY Growth %',
        yaxis_title='%', height=350, margin=dict(l=50, r=20, t=50, b=40)
    )
    print('Additional charts built.')


In [ ]:
# -- 6. Country Scoreboard (HTML tables, one per basket) -------------------
sb = pd.read_parquet(OUTPUTS_DIR / "country_scoreboard.parquet")


def _bg(v, lo, hi, invert=False):
    try:
        f = float(v)
        if np.isnan(f):
            return ""
    except Exception:
        return ""
    if not invert:
        if f >= hi: return "background:rgba(102,187,106,0.30)"
        if f >= lo: return "background:rgba(255,152,0,0.30)"
        return "background:rgba(239,83,80,0.30)"
    else:
        if f <= lo: return "background:rgba(102,187,106,0.30)"
        if f <= hi: return "background:rgba(255,152,0,0.30)"
        return "background:rgba(239,83,80,0.30)"


def _cell(row, metric, spec, lo=None, hi=None, invert=False):
    """One scoreboard <td>. Three states, never blank (specs/006 US3):
      - value + fresh  -> value, title shows the as-of date
      - value + stale  -> muted value + dagger, title explains it's past the window
      - not available  -> explicit N/A cell (distinct from stale)"""
    v = row.get(metric)
    try:
        fv = float(v)
        missing = np.isnan(fv)
    except (TypeError, ValueError):
        missing = True
    if missing:
        return '<td class="sb-na" title="not reported for this economy">N/A</td>'

    asof = row.get(f"{metric}_as_of")
    stale = bool(row.get(f"{metric}_stale"))
    style = _bg(fv, lo, hi, invert) if lo is not None else ""
    cls = "sb-stale" if stale else ""
    if asof:
        title = f"as of {asof}" + (" - older than this basket's freshness window" if stale else "")
    else:
        title = "as-of date unavailable"
    mark = '<sup class="stale-mark">&dagger;</sup>' if stale else ""
    return f'<td class="{cls}" style="{style}" title="{title}">{spec.format(fv)}{mark}</td>'


def _basket_table(part):
    """Render one <table> for a single basket's rows."""
    rows_html = []
    for country, row in part.iterrows():
        cells = "".join([
            f'<td class="cn">{country}</td>',
            _cell(row, "gdp_actual",      "{:+.1f}",  0,  2),
            _cell(row, "gdp_forecast",    "{:+.1f}",  0,  2),
            _cell(row, "inflation",       "{:.1f}",   3,  5, True),
            _cell(row, "unemployment",    "{:.1f}",   5,  8, True),
            _cell(row, "current_account", "{:+.1f}", -3,  0),
            _cell(row, "govt_debt",       "{:.0f}",  60, 90, True),
            _cell(row, "policy_rate",     "{:.2f}"),
            _cell(row, "stock_ytd",       "{:+.1f}", -10, 0),
        ])
        rows_html.append(f"<tr>{cells}</tr>")
    return (
        '<table class="sb"><thead><tr>'
        '<th>Country</th><th>GDP %<br>Actual</th><th>GDP %<br>Forecast</th>'
        '<th>CPI %</th><th>Unemp %</th><th>Curr Acct<br>% GDP</th>'
        '<th>Govt Debt<br>% GDP</th><th>Policy<br>Rate %</th><th>Stock<br>YTD %</th>'
        f'</tr></thead><tbody>{"".join(rows_html)}</tbody></table>'
    )


# Ordered baskets; only render those actually present in the data.
BASKET_ORDER = [
    ("Major", "Major Economies"),
    ("Emerging Markets", "Emerging Markets"),
    ("Developing", "Developing &amp; Frontier"),
]
_basket_col = sb["basket"] if "basket" in sb.columns else None

sections = []
for key, label in BASKET_ORDER:
    part = sb[_basket_col == key] if _basket_col is not None else (sb if key == "Major" else sb.iloc[0:0])
    if part.empty:
        continue
    sections.append(f'<h3 class="sb-h">{label}</h3>{_basket_table(part)}')

scoreboard_html = f"""
<style>
.sb {{width:100%;border-collapse:collapse;font-family:monospace;font-size:13px;margin-bottom:6px;}}
.sb th {{background:#313244;color:#cdd6f4;padding:9px 14px;text-align:center;
         border:1px solid #45475a;font-size:11px;line-height:1.5;}}
.sb td {{color:#cdd6f4;padding:7px 14px;text-align:center;border:1px solid #1e1e2e;}}
.sb .cn {{text-align:left;background:#1e1e2e;font-weight:500;white-space:nowrap;padding-left:16px;}}
.sb .sb-na {{color:#6c7086;font-style:italic;}}
.sb .sb-stale {{opacity:0.55;}}
.sb .stale-mark {{color:#f9e2af;font-size:10px;margin-left:1px;}}
.sb tr:hover td {{filter:brightness(1.2);}}
.sb-h {{color:#cdd6f4;font-family:sans-serif;font-size:15px;font-weight:600;
        margin:22px 2px 8px;padding-bottom:4px;border-bottom:1px solid #45475a;}}
.sb-h:first-of-type {{margin-top:4px;}}
.sb-legend {{color:#9399b2;font-size:11px;font-family:monospace;margin:10px 2px 0;line-height:1.5;}}
.sb-legend .stale-mark {{color:#f9e2af;}}
</style>
{"".join(sections)}
<p class="sb-legend"><sup class="stale-mark">&dagger;</sup> = data older than this basket's freshness window &middot;
hover any figure for its as-of date &middot; N/A = not reported.
Freshness windows widen for less-timely baskets (Emerging Markets and Developing report on a longer lag).</p>"""

print("Scoreboard HTML built for baskets:", [k for k, _ in BASKET_ORDER if (_basket_col == k).any()] if _basket_col is not None else ["Major"])
_show = [c for c in sb.columns if not c.endswith("_as_of") and not c.endswith("_stale")]
print(sb[_show].to_string())

In [ ]:
# -- Assemble HTML dashboard with all charts + footer -----------------
curve_html  = fig_curve.to_html(full_html=False, include_plotlyjs='cdn')
rec_html    = fig_rec.to_html(full_html=False,   include_plotlyjs=False)
inf_html    = fig_inf.to_html(full_html=False,   include_plotlyjs=False)
gauge_html  = fig_gauge.to_html(full_html=False, include_plotlyjs=False)

# Financial-conditions snapshot helpers
def _fv(v, fmt='.2f', sign=False, suffix=''):
    if v is None: return 'N/A'
    fs = f'{v:+{fmt}}' if sign else f'{v:{fmt}}'
    return fs + suffix

def _fc(v, green_t, red_t, invert=False):
    G, R, O = '#66bb6a', '#ef5350', '#ff9800'
    if v is None: return '#a6adc8'
    if not invert:
        return G if v > green_t else (R if v < red_t else O)
    return G if v < green_t else (R if v > red_t else O)

ext_metrics_html = f"""
<div style='font-family:monospace;background:#1e1e2e;color:#cdd6f4;
            padding:24px;border-radius:8px;line-height:2'>
  <h2 style='color:#89b4fa;margin-top:0;font-size:1rem;letter-spacing:.06em'>
      FINANCIAL CONDITIONS</h2>
  <table style='width:100%;border-collapse:collapse'>
    <tr><td>📊 HY Credit Spread</td>
        <td style='color:{_fc(credit_spread_val, 300, 500, invert=True)}'>
        {_fv(credit_spread_val, '.0f', suffix=' bps')}</td></tr>
    <tr><td>📉 Real 10y Rate</td>
        <td style='color:{_fc(real_rate_val, 0, -1)}'>
        {_fv(real_rate_val, '.2f', sign=True, suffix='%')}</td></tr>
    <tr><td>🏦 Fed Funds Rate</td>
        <td style='color:#a6adc8'>{_fv(fed_funds_val, '.2f', suffix='%')}</td></tr>
    <tr><td>👷 Unemployment Rate</td>
        <td style='color:{_fc(unrate_val, 4.5, 6.5, invert=True)}'>
        {_fv(unrate_val, '.1f', suffix='%')}</td></tr>
    <tr><td>💰 M2 YoY Growth</td>
        <td style='color:{_fc(m2_yoy_val, 0, -5)}'>
        {_fv(m2_yoy_val, '.1f', sign=True, suffix='%')}</td></tr>
    <tr><td>📈 CPI YoY</td>
        <td style='color:{_fc(cpi_yoy_val, 3.5, 5.5, invert=True) if cpi_yoy_val is not None else "#a6adc8"}'>
        {_fv(cpi_yoy_val, '.1f', suffix='%')}</td></tr>
  </table>
</div>"""

# Additional chart HTML (guarded)
if _us_ok:
    credit_html = fig_credit.to_html(full_html=False, include_plotlyjs=False)
    real_html   = fig_real.to_html(full_html=False,   include_plotlyjs=False)
    fed_html    = fig_fed.to_html(full_html=False,    include_plotlyjs=False)
    m2_html     = fig_m2.to_html(full_html=False,     include_plotlyjs=False)
    extra_grid  = (f'<div class="card">{credit_html}</div>'
                   f'<div class="card">{real_html}</div>'
                   f'<div class="card">{fed_html}</div>'
                   f'<div class="card">{m2_html}</div>')
else:
    extra_grid = ''

html = f"""<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='UTF-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1.0'>
  <title>Global Macro Dashboard</title>
  <style>
    body     {{ margin:0; background:#11111b; color:#cdd6f4; font-family:system-ui,sans-serif; }}
    h1       {{ text-align:center; color:#89b4fa; padding:24px 0 0; margin:0; font-size:1.6rem; }}
    p.sub    {{ text-align:center; color:#6c7086; margin:4px 0 0; font-size:.9rem; }}
    .tab-bar {{ display:flex; gap:0; padding:16px 16px 0; border-bottom:2px solid #313244; }}
    .tab     {{ background:transparent; color:#a6adc8; border:none;
               border-bottom:2px solid transparent; margin-bottom:-2px;
               padding:10px 28px; cursor:pointer; font-size:.95rem;
               font-family:system-ui,sans-serif; }}
    .tab:hover  {{ color:#cdd6f4; }}
    .tab.active {{ color:#89b4fa; border-bottom-color:#89b4fa; }}
    .tab-pane   {{ display:none; }}
    .tab-pane.active {{ display:block; }}
    .grid    {{ display:grid; grid-template-columns:1fr 1fr; gap:16px; padding:16px; }}
    .card    {{ background:#1e1e2e; border-radius:8px; padding:8px; }}
    .full    {{ grid-column:1/-1; }}
    .sb-wrap {{ padding:16px 24px 24px; overflow-x:auto; }}
    .legend  {{ display:flex; gap:20px; justify-content:flex-end;
               padding:12px 24px 0; font-size:.8rem; color:#6c7086; align-items:center; }}
    .dot     {{ width:10px; height:10px; border-radius:50%;
               display:inline-block; margin-right:5px; vertical-align:middle; }}
    footer   {{ text-align:center; padding:20px 0 32px; font-size:.8rem;
               color:#6c7086; border-top:1px solid #313244; margin-top:8px; }}
    footer a {{ color:#89b4fa; text-decoration:none; }}
  </style>
</head>
<body>
  <h1>Global Macro Dashboard</h1>
  <p class='sub'>As of {snap['as_of']} &middot; Powered by FRED &middot; World Bank &middot; IMF WEO</p>

  <div class='tab-bar'>
    <button class='tab active' onclick="switchTab(event,'macro')">Macro Signals</button>
    <button class='tab' onclick="switchTab(event,'scoreboard')">Country Scoreboard</button>
  </div>

  <div id='macro' class='tab-pane active'>
    <div class='grid'>
      <div class='card full'>{metrics_html}</div>
      <div class='card full'>{ext_metrics_html}</div>
      <div class='card'>{curve_html}</div>
      <div class='card'>{rec_html}</div>
      <div class='card'>{inf_html}</div>
      <div class='card'>{gauge_html}</div>
      {extra_grid}
    </div>
  </div>

  <div id='scoreboard' class='tab-pane'>
    <div class='legend'>
      <span><span class='dot' style='background:rgba(102,187,106,0.9)'></span>Positive</span>
      <span><span class='dot' style='background:rgba(255,152,0,0.9)'></span>Caution</span>
      <span><span class='dot' style='background:rgba(239,83,80,0.9)'></span>Negative</span>
    </div>
    <div class='sb-wrap'>{scoreboard_html}</div>
  </div>

  <footer>
    <a href='paper.html'>Methodology Paper</a>
    &nbsp;&middot;&nbsp;
    <a href='newsletter/'>Newsletter Archive</a>
    &nbsp;&middot;&nbsp;
    <a href='https://github.com/trevmon28/macro-dashboard'>GitHub</a>
  </footer>

  <script>
    function switchTab(e, id) {{
      document.querySelectorAll('.tab-pane').forEach(p => p.classList.remove('active'));
      document.querySelectorAll('.tab').forEach(b => b.classList.remove('active'));
      document.getElementById(id).classList.add('active');
      e.currentTarget.classList.add('active');
    }}
  </script>
</body>
</html>"""

out = DOCS_DIR / 'index.html'
out.write_text(html, encoding='utf-8')
print(f'Dashboard saved: {out} ({len(html):,} bytes)')


In [ ]:
# Push docs/index.html to GitHub (Colab only — Actions handles this via the workflow)
# Reads GITHUB_TOKEN from Colab Secrets — never paste tokens in code.
# To add the secret: click the key icon (🔑) in the Colab left sidebar
# → New secret → Name: GITHUB_TOKEN, Value: your token
if not _IN_COLAB:
    print("Skipping GitHub push — running in Actions, workflow handles commits.")
else:
    import requests, base64

    REPO   = "trevmon28/macro-dashboard"
    BRANCH = "master"
    FILE   = "docs/index.html"

    try:
        from google.colab import userdata
        TOKEN = userdata.get("GITHUB_TOKEN")
        if not TOKEN:
            raise ValueError("GITHUB_TOKEN secret is empty")
    except Exception as e:
        raise EnvironmentError(
            "Add GITHUB_TOKEN to Colab Secrets: key icon → New secret → GITHUB_TOKEN"
        ) from e

    headers = {
        "Authorization": f"token {TOKEN}",
        "Accept": "application/vnd.github.v3+json",
    }

    content_b64 = base64.b64encode(out.read_bytes()).decode()

    # Get current SHA
    r = requests.get(
        f"https://api.github.com/repos/{REPO}/contents/{FILE}?ref={BRANCH}",
        headers=headers,
    )
    r.raise_for_status()
    sha = r.json()["sha"]

    # Push
    payload = {
        "message": f"dashboard: refresh {snap['as_of']}",
        "content": content_b64,
        "sha": sha,
        "branch": BRANCH,
    }
    r = requests.put(
        f"https://api.github.com/repos/{REPO}/contents/{FILE}",
        json=payload,
        headers=headers,
    )
    r.raise_for_status()
    print("Pushed:", r.json()["commit"]["sha"])
    print("Dashboard will update at https://trevmon28.github.io/macro-dashboard/ in ~60s")